## Data Agent Source Update

**The code in this notebook was generated with copilot for quick PoC and should be reviewed prior to deployment in any environment.**

This notebook is designed to change the source that a data agent points to since it cannot be updated directly in a deployment pipeline.

Specifically this notebook updates a single semantic model source for a Data Agent, however it can be updated to include other outcomes.

For this example we have 2 distinct pipelines;
1. Semantic Model pipeline that contains dev and test workspaces for the semantic model that serves as the datasource for the Data Agent item
2. Data Agent pipeline that serves as a pipeline to move a Data Agent item through the development lifecycle.  For this example we include only a Dev and Test workspace.

### Parameters
We paramaterize the Workspaces and objects required as well as add a bearer token for issuing our calls


In [ ]:

# ===== PARAMETERS =====
TENANT_ID = ""  # your tenant ID
WORKSPACE_ID = "" #workspace ID that contains the Data Agent
AGENT_ID = "" #Data agent ID
NEW_DATA_SOURCE_ID = "" #ID of semantic model that you want Agent to point to after update
NEW_DATA_SOURCE_WORKSPACE_ID = "" #workspace where semantic model above sits

# Optional: set to True to print full agent definition
DEBUG = True

#insert your own Bearer token here
access_token = "" #your token

 #create the headers object for authenticating HTTP calls.
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

StatementMeta(, 128e4075-af99-4142-86e2-252ab0d98ad3, 33, Finished, Available, Finished, False)

### SPN Access
If you prefer to use SPN you can leverage the below code with client ID and Secret ID to obtain a token instead.

In [ ]:
import requests

def get_fabric_token(tenant_id):
    token_url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    
    payload = {
        "grant_type": "client_credentials",
        "client_id": mssparkutils.credentials.getSecret("<keyvault>", "<client-id-secret>"),
        "client_secret": mssparkutils.credentials.getSecret("<keyvault>", "<client-secret>"),
        "scope": "https://analysis.windows.net/powerbi/api/.default"
    }

    response = requests.post(token_url, data=payload)
    response.raise_for_status()
    return response.json()["access_token"]

access_token = get_fabric_token(TENANT_ID)

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

### Get Existing Agent Definition

This is a long running operation.  We expect a 202 response and having to poll for success.  

Once success is obtained, we use the operation ID to retrieve the existing definition as a json payload

In [25]:
import time
import requests

def get_agent_definition(headers, workspace_id, agent_id, poll_interval_sec=2):
    """
    Fetch agent definition using Fabric getDefinition async API.
    """

    # 1️⃣ Start async getDefinition operation
    start_url = (
        f"https://api.fabric.microsoft.com/v1/workspaces/"
        f"{workspace_id}/dataAgents/{agent_id}/getDefinition"
    )

    start_response = requests.post(start_url, headers=headers)
    start_response.raise_for_status()

    if start_response.status_code != 202:
        raise Exception(
            f"Unexpected status from getDefinition kickoff: "
            f"{start_response.status_code} - {start_response.text}"
        )

    operation_location = start_response.headers.get("Location")
    operationId = start_response.headers.get("x-ms-operation-id")
    if not operation_location:
        raise Exception("Missing Location header from getDefinition response")

    # 2️⃣ Poll operation
    while True:
        poll_response = requests.get(operation_location, headers=headers)
        poll_response.raise_for_status()

        poll_payload = poll_response.json()
        status = poll_payload.get("status")

        if status == "Succeeded":
            break
        elif status in ("Failed", "Canceled"):
            raise Exception(f"getDefinition failed: {poll_payload}")

        time.sleep(poll_interval_sec)

    # 3️⃣ Fetch definition from definitionLocation
    definition_location = f"https://api.fabric.microsoft.com/v1/operations/{operationId}/result"
    print (definition_location)
    if not definition_location:
        raise Exception(
            f"getDefinition succeeded but no definitionLocation returned: {poll_payload}"
        )

    definition_response = requests.get(definition_location, headers=headers)
    definition_response.raise_for_status()

    return definition_response.json()


StatementMeta(, 128e4075-af99-4142-86e2-252ab0d98ad3, 27, Finished, Available, Finished, False)

### View Result

In [ ]:


agent_definition = get_agent_definition(
    headers=headers,
    workspace_id=WORKSPACE_ID,
    agent_id=AGENT_ID
)

if DEBUG:
    import json
    print(json.dumps(agent_definition, indent=2))


### Decode the payload

Payload is delivered in Base64 encoding

In [ ]:
import base64, json

def decode_inline_payload(part):
    raw = base64.b64decode(part["payload"])
    return json.loads(raw.decode("utf-8"))

# find the datasource.json part
datasource_part = next(
    p for p in agent_definition["definition"]["parts"]
    if p["path"].endswith("/datasource.json")
)

datasource_json = decode_inline_payload(datasource_part)

datasource_json


### Update existing definition

We need to replace the source data; specifically the workspace ID and the Semantic Model ID of the model we want to point at.

In [32]:

NEW_DATASET_ID = NEW_DATA_SOURCE_ID
NEW_WORKSPACE_ID = NEW_DATA_SOURCE_WORKSPACE_ID

datasource_json["artifactId"] = NEW_DATASET_ID
datasource_json["workspaceId"] = NEW_WORKSPACE_ID


StatementMeta(, 128e4075-af99-4142-86e2-252ab0d98ad3, 34, Finished, Available, Finished, False)

### Encode our payload
After updating our json we can re-encode the payload to provide back to Fabric as the UpdateDefinition call.

In [33]:

def encode_inline_payload(obj):
    text = json.dumps(obj, ensure_ascii=False, indent=2)
    return base64.b64encode(text.encode("utf-8")).decode("ascii")

datasource_part["payload"] = encode_inline_payload(datasource_json)
datasource_part["payloadType"] = "InlineBase64"


StatementMeta(, 128e4075-af99-4142-86e2-252ab0d98ad3, 35, Finished, Available, Finished, False)

In [34]:
import time
import requests

def poll_fabric_operation(headers, operation_id=None, operation_url=None, default_sleep=5, timeout_seconds=600):
    """
    Polls Fabric LRO until terminal state.
    You can pass operation_url (Location header) or operation_id (x-ms-operation-id).
    Uses GET /v1/operations/{operationId}. 【2-082031】
    """
    if not operation_url:
        if not operation_id:
            raise ValueError("Provide either operation_url or operation_id")
        operation_url = f"https://api.fabric.microsoft.com/v1/operations/{operation_id}"

    start = time.time()
    last_payload = None

    while True:
        resp = requests.get(operation_url, headers=headers)
        resp.raise_for_status()
        payload = resp.json()
        last_payload = payload

        status = payload.get("status")
        if status in ("Succeeded", "Failed", "Canceled"):
            return payload

        # Respect Retry-After when present; otherwise sleep default
        retry_after = resp.headers.get("Retry-After")
        sleep_s = int(retry_after) if retry_after and retry_after.isdigit() else default_sleep

        if time.time() - start > timeout_seconds:
            raise TimeoutError(f"Operation timed out after {timeout_seconds}s. Last payload: {last_payload}")

        time.sleep(sleep_s)


def get_operation_result(headers, operation_id):
    """
    Fetch LRO result from /v1/operations/{operationId}/result 【2-082031】
    """
    url = f"https://api.fabric.microsoft.com/v1/operations/{operation_id}/result"
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    return resp.json()


def update_data_agent_definition(headers, workspace_id, data_agent_id, definition_parts_payload,
                                update_metadata=False, timeout_seconds=600):
    """
    Calls Data Agent Update Definition:
      POST /v1/workspaces/{workspaceId}/dataAgents/{dataAgentId}/updateDefinition 【1-a9dd28】

    definition_parts_payload should be either:
      - the full object { "definition": { "parts": [...] } }
        OR
      - just { "parts": [...] } (we will wrap it)

    Returns:
      - If synchronous (200): response.json()
      - If async (202): final operation state payload + (best-effort) operation result if available
    """

    # Ensure we have the correct request body shape expected by the API 【1-a9dd28】
    if "definition" in definition_parts_payload:
        body = {"definition": definition_parts_payload["definition"]}
    elif "parts" in definition_parts_payload:
        body = {"definition": {"parts": definition_parts_payload["parts"]}}
    else:
        raise ValueError("definition_parts_payload must contain either 'definition' or 'parts'")

    # Build endpoint (optional updateMetadata query param) 【1-a9dd28】
    url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/dataAgents/{data_agent_id}/updateDefinition"
    if update_metadata:
        url += "?updateMetadata=true"

    resp = requests.post(url, headers=headers, json=body)

    # 200: completed immediately 【1-a9dd28】
    if resp.status_code == 200:
        return {
            "mode": "sync",
            "status_code": 200,
            "response": resp.json()
        }

    # 202: long-running operation started; headers include Location + x-ms-operation-id 【1-a9dd28】【2-082031】
    if resp.status_code == 202:
        operation_url = resp.headers.get("Location")
        operation_id = resp.headers.get("x-ms-operation-id")

        if not operation_url and not operation_id:
            raise Exception(f"202 returned but missing Location/x-ms-operation-id headers. Headers: {dict(resp.headers)}")

        # Poll until terminal status 【2-082031】
        final_state = poll_fabric_operation(
            headers=headers,
            operation_id=operation_id,
            operation_url=operation_url,
            timeout_seconds=timeout_seconds
        )

        result_payload = None

        # If succeeded, try to fetch /result (some ops provide it) 【2-082031】
        if final_state.get("status") == "Succeeded" and operation_id:
            try:
                result_payload = get_operation_result(headers, operation_id)
            except Exception:
                # Not all operations provide a result payload; ignore if absent/unsupported 【2-082031】
                result_payload = None

        return {
            "mode": "async",
            "status_code": 202,
            "operation_id": operation_id,
            "operation_url": operation_url,
            "final_state": final_state,
            "result": result_payload
        }

    # Any other status: raise with details
    try:
        resp.raise_for_status()
    except Exception as e:
        raise Exception(f"updateDefinition failed: {resp.status_code} {resp.text}") from e

StatementMeta(, 128e4075-af99-4142-86e2-252ab0d98ad3, 36, Finished, Available, Finished, False)

### Update the Agent Definition

In [37]:
# agent_def is the JSON you currently have in memory (the one with "definition": {"parts":[...]} )
# e.g., agent_def = agent_definition_result_from_getDefinition

UPDATE_METADATA = False  # set True only if you WANT .platform metadata applied 【1-a9dd28】

result = update_data_agent_definition(
    headers=headers,
    workspace_id=WORKSPACE_ID,
    data_agent_id=AGENT_ID,
    definition_parts_payload=agent_definition,     # <-- pass the whole object
    update_metadata=UPDATE_METADATA,
    timeout_seconds=900
)

result

StatementMeta(, 128e4075-af99-4142-86e2-252ab0d98ad3, 39, Finished, Available, Finished, False)

{'mode': 'async',
 'status_code': 202,
 'operation_id': 'dbc197f1-bb96-4ceb-9a11-742d6890956e',
 'operation_url': 'https://wabi-west-us3-a-primary-redirect.analysis.windows.net/v1/operations/dbc197f1-bb96-4ceb-9a11-742d6890956e',
 'final_state': {'status': 'Succeeded',
  'createdTimeUtc': '2026-04-06T21:42:17.6623086',
  'lastUpdatedTimeUtc': '2026-04-06T21:42:19.9970571',
  'percentComplete': 100,
  'error': None},
 'result': {'id': 'c79984c1-269c-4905-8a58-e9413c988ca5',
  'type': 'DataAgent',
  'displayName': 'covid_agent',
  'description': '',
  'workspaceId': '49e25688-9e95-444b-b320-168899a192a5'}}